# Comments

comments here

# Init & functions

In [1]:
from src.prompt_manager import PromptManager, PromptSuite, PromptTemplate, create_empty_prompt_template, create_chat_prompt_dict
from pathlib import Path

In [11]:
# OLD BUILD FUNCTION

def build_formative_suite_old(config: dict, suite_description: str) -> PromptSuite:
    """
    Parses a configuration dictionary to build a PromptSuite for psychometric evaluation.
    """
    prompt_templates = []

    for field_key, field_config in config.items():
        human_name = field_config["human_name"]
        user_template = field_config["user_template"]
        base_tags = field_config["tags"]

        for dim_key, dim_data in field_config["dimensions"].items():

            # 1. Construct the System Prompt
            system_prompt = f"""You are an expert evaluator of influencer marketing deals. 
Your task is to evaluate the provided {human_name} conditional strictly on its {dim_key.replace('_', ' ')}.
This measures {dim_data['definition']}

Scale definition:
{dim_data['rubric']}

Output only the integer score."""

            # 2. Assemble the Template Dictionary
            templ_dict = create_chat_prompt_dict(
                name=f"rubric_{field_key}_{dim_key}",
                description=f"Evaluates the intensity of {dim_key} on the {human_name}",
                template_chat={
                    "system": system_prompt.strip(),
                    "user": user_template.strip()
                },
                token_constraints=["1", "2", "3", "4"],
                template_tags=['rubric', 'scale_4',
                               'trait_intensity', dim_key] + base_tags,
                dimension_name=f"{field_key}_{dim_key}"
            )

            # 3. Append to list
            prompt_templates.append(PromptTemplate.from_dict(templ_dict))

    # Return the fully compiled suite
    metadata = {'description': suite_description}
    return PromptSuite.from_list(prompt_templates, metadata=metadata)


# NEW BUILD PROMPT

def build_formative_suite(
    config: dict,
    suite_description: str,
    scale_size: int,
    system_prompt_template: str
) -> PromptSuite:
    """
    Parses a configuration dictionary to build a PromptSuite for psychometric evaluation.
    Utilizes .format() to dynamically inject variables into interchangeable system prompt templates.
    """
    prompt_templates = []

    # Dynamically generate the scale tag (e.g., "scale_7")
    dynamic_scale_tag = f"scale_{scale_size}"

    for field_key, field_config in config.items():
        human_name = field_config["human_name"]
        user_template = field_config["user_template"]
        base_tags = field_config["tags"]

        for dim_key, dim_data in field_config["dimensions"].items():

            # Clean the dimension key for the prompt (e.g., "creator_brand_alignment" -> "creator brand alignment"), and capitalize
            dim_key_clean = dim_key.replace('_', ' ').title()

            # 1. Construct the System Prompt via .format()
            system_prompt = system_prompt_template.format(
                human_name=human_name,
                dim_key_clean=dim_key_clean,
                definition=dim_data['definition'],
                rubric=dim_data['rubric']
            )

            # 2. Assemble the Template Dictionary
            templ_dict = create_chat_prompt_dict(
                name=f"rubric_{field_key}_{dim_key}",
                description=f"Evaluates the intensity of {dim_key} on the {human_name}",
                template_chat={
                    "system": system_prompt.strip(),
                    "user": user_template.strip()
                },
                # Create a new list object in memory for each iteration to avoid YAML aliasing
                token_constraints=[str(i) for i in range(1, scale_size + 1)],
                template_tags=['rubric', dynamic_scale_tag,
                               'trait_intensity', dim_key] + base_tags,
                dimension_name=f"{field_key}_{dim_key}"
            )

            # 3. Append to list
            prompt_templates.append(PromptTemplate.from_dict(templ_dict))

    # Return the fully compiled suite
    metadata = {'description': suite_description}
    return PromptSuite.from_list(prompt_templates, metadata=metadata)

# __Barter deals__

In [3]:
suite_folder = Path("../prompts/PromptSuites/sandbox/BARTER_DEALS")
pm = PromptManager(suite_folder)

PromptManager initialized with folder: ..\prompts\PromptSuites\sandbox\BARTER_DEALS


In [18]:
SYSTEM_PROMPT_OLD = """You are an expert evaluator of influencer marketing deals. 
Your task is to evaluate the provided {human_name} conditional strictly on its {dim_key_clean}.
This measures {definition}

Scale definition:
{rubric}

Output only the integer score."""

SYSTEM_PROMPT_NEW = """You are an expert evaluator of influencer marketing deals for Barter, an app-based creator marketplace. 
In this ecosystem, creators scroll a feed of deals and apply to collaborate. If accepted, they produce content (e.g., UGC, TikToks, Reels) primarily in exchange for the brand's product or service, with occasional cash supplements. Your evaluation must be grounded in this specific, low-friction, product-driven marketplace dynamic.

Your task is to evaluate the provided {human_name} conditional strictly on its {dim_key_clean}.
{definition}

Scale definition:
{rubric}

Output only the integer score."""


SYSTEM_PROMPT_NEW2 = """You are an expert evaluator of influencer marketing deals ("Deal Texts") for Barter, an app-based creator marketplace. 
In this ecosystem, creators scroll a feed of deals and apply to collaborate. If accepted, they produce content (e.g., UGC, TikToks, Instagram Reels) primarily in exchange for the brand's offering, with occasional cash supplements. Your evaluation must be grounded in this creator ecosystem.

You will be provided with a Deal Text consisting of a Title, Body, and Requirements. Please analyze this text, focusing strictly on the following dimension: **{dim_key_clean}**.

Dimension Definition:
{definition}

Scale Definition:
{rubric}

Evaluate where the text falls on this continuum, and provide the closest whole number rating (1-7). """

## Formative

### Continuous anchors, no BARS Formative

Here, we experiment with dropping BARS in favor of more generic anchors, and leave the construct definition entirely out of the anchors. We opt for this, because previously anchors were quite specific descriptions; this way, it became more of a pattern matching task, with the major risk that if some pattern is very recognizable, it just gets bucketed into one level of the rating scale. Instead, we should use the fact that language carries continuity semantically; that is, words like "weak, fair, good, excellent" cover a continuous space semantically. 



#### Scale size 4

In [ ]:
EVALUATION_CONFIG = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "brand_recognition": {
                "definition": "You are measuring 'Brand Recognition.' This construct evaluates the inherent brand equity and public awareness of the specific brand or company, independent of the actual product category. It measures the continuum between a completely anonymous, generic, or newly founded entity with zero existing social proof, up to a universally recognized, ubiquitous household name that instantly lends massive mainstream credibility to the creator.",
                "rubric": "1: Weak\n2: Fair\n3: Good\n4: Excellent"
            },
            "creator_brand_alignment": {
                "definition": "You are measuring 'Creator Brand Alignment.' This construct evaluates the degree to which the product or service naturally integrates into the aspirational aesthetic and lifestyle identities typical of content creators (e.g., beauty, fashion, culinary, travel, fitness). It measures the continuum between a dry, highly corporate, or mundane utility product that inherently clashes with a creator's personal brand, up to a highly photogenic, trendy, and aspirational product that actively elevates it.",
                "rubric": "1: Weak\n2: Fair\n3: Good\n4: Excellent"
            },
            "value_proposition_clarity": {
                "definition": "You are measuring 'Value Proposition Clarity.' This construct evaluates how explicitly the text communicates the exact reward and core benefit of the collaboration to the creator. It measures the continuum between a highly ambiguous, buried, or obscured value exchange requiring the reader to guess the benefits, up to a highly transparent, sharply formatted pitch where the precise rewards and collaboration terms are immediately obvious and focal.",
                "rubric": "1: Weak\n2: Fair\n3: Good\n4: Excellent"
            },
            "call_to_action_strength": {
                "definition": "You are measuring 'Call to Action Strength.' This construct evaluates the urgency, explicitness, and compelling nature of the prompt directing the creator's next steps. It measures the continuum between a completely passive, absent, or heavily implied prompt with no clear direction, up to an intensely compelling, urgent, and direct call to action that clearly dictates the next step and drives immediate excitement.",
                "rubric": "1: Weak\n2: Fair\n3: Good\n4: Excellent"
            },
            "product_universality": {
                "definition": "You are measuring 'Product Universality.' This construct evaluates the breadth of the product's target demographic. It measures the continuum between a highly specialized, restrictive product requiring specific hobbies or niche traits, up to a highly accessible, everyday product that appeals broadly to the general mass market.",
                "rubric": "1: Weak\n2: Fair\n3: Good\n4: Excellent"
            },
        }
    },
    "creators_requirement": {
        "human_name": "Creator Requirements",
        "user_template": "Creator Requirements:\n{creators_requirement}",
        "tags": ["creators_requirement"],
        "dimensions": {
            "workload_volume": {
                "definition": "You are measuring 'Workload Volume.' This construct evaluates the sheer magnitude of content creation, deliverables, and ongoing effort demanded from the creator. It measures the continuum between a virtually frictionless, minimal-effort request (e.g., a single simple story or quick amplification), up to an excessively high-friction commitment requiring massive ongoing deliverable volumes, complex production, or intensive long-term coordination.",
                "rubric": "1: Very Low\n2: Low\n3: High\n4: Very High"
            }
        }
    }
}

In [ ]:
# # Build and save the suite
# diagnostic_suite = build_formative_suite_old(
#     EVALUATION_CONFIG, "Prompt suite with continuous anchors (No BARS) V2")
# diagnostic_suite.save(suite_folder, filename="continuous_noBARS_V2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\continuous_noBARS_V2_suite_d39649b54a27.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/continuous_noBARS_V2_suite_d39649b54a27.yml')

In [56]:
# Build and save the suite
diagnostic_suite = build_formative_suite(
    EVALUATION_CONFIG, "Prompt suite with continuous anchors (No BARS) V2", 4, SYSTEM_PROMPT_OLD)
diagnostic_suite.save(suite_folder, filename="continuous_noBARS_V2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\continuous_noBARS_V2_suite_d39649b54a27.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/continuous_noBARS_V2_suite_d39649b54a27.yml')

#### Scale size 7

We increase the scale size for 'semantic zooming': we see that the scales are dominated by ratings in certain levels of the scale, e.g., a lot of deals getting a 2 rating. We need to increase the variance around these ratings.

In [57]:
EVALUATION_CONFIG = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "creator_brand_alignment": {
                "definition": "You are measuring 'Creator Brand Alignment.' This construct evaluates the degree to which the product or service naturally integrates into the aspirational aesthetic and lifestyle identities typical of content creators (e.g., beauty, fashion, culinary, travel, fitness). It measures the continuum between a dry, highly corporate, or mundane utility product that inherently clashes with a creator's personal brand, up to a highly photogenic, trendy, and aspirational product that actively elevates it.",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Fair\n4: Moderate\n5: Good\n6: Very Good\n7: Excellent"
            },

            "call_to_action_strength": {
                "definition": "You are measuring 'Call to Action Strength.' This construct evaluates the urgency, explicitness, and compelling nature of the prompt directing the creator's next steps. It measures the continuum between a completely passive, absent, or heavily implied prompt with no clear direction, up to an intensely compelling, urgent, and direct call to action that clearly dictates the next step and drives immediate excitement.",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Fair\n4: Moderate\n5: Good\n6: Very Good\n7: Excellent"
            },

            "product_universality": {
                "definition": "You are measuring 'Product Universality.' This construct evaluates the breadth of the product's target demographic. It measures the continuum between a highly specialized, restrictive product requiring specific hobbies or niche traits, up to a highly accessible, everyday product that appeals broadly to the general mass market.",
                "rubric": "1: Extremely Narrow\n2: Narrow\n3: Somewhat Narrow\n4: Moderate\n5: Somewhat Broad\n6: Broad\n7: Extremely Broad"
            }
        }
    }
}

In [ ]:
# Old system prompt
diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG,
    suite_description="Prompt suite with continuous anchors (No BARS), scale 7",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_OLD)
diagnostic_suite.save(suite_folder, filename="continuous_noBARS_scale7")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\continuous_noBARS_scale7_suite_d23aac70a751.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/continuous_noBARS_scale7_suite_d23aac70a751.yml')

In [ ]:
# New system prompt
diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG,
    suite_description="Prompt suite with continuous anchors (No BARS), scale 7, system prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
diagnostic_suite.save(suite_folder, filename="continuous_noBARS_scale7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\continuous_noBARS_scale7_SPV2_suite_c346be6c6bbc.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/continuous_noBARS_scale7_SPV2_suite_c346be6c6bbc.yml')

- Including requirements

In [ ]:
EVALUATION_CONFIG = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "creator_brand_alignment": {
                "definition": "You are measuring 'Creator Brand Alignment.' This construct evaluates the degree to which the product or service naturally integrates into the aspirational aesthetic and lifestyle identities typical of content creators (e.g., beauty, fashion, culinary, travel, fitness). It measures the continuum between a dry, highly corporate, or mundane utility product that inherently clashes with a creator's personal brand, up to a highly photogenic, trendy, and aspirational product that actively elevates it.",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Fair\n4: Moderate\n5: Good\n6: Very Good\n7: Excellent"
            },

            "call_to_action_strength": {
                "definition": "You are measuring 'Call to Action Strength.' This construct evaluates the urgency, explicitness, and compelling nature of the prompt directing the creator's next steps. It measures the continuum between a completely passive, absent, or heavily implied prompt with no clear direction, up to an intensely compelling, urgent, and direct call to action that clearly dictates the next step and drives immediate excitement.",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Fair\n4: Moderate\n5: Good\n6: Very Good\n7: Excellent"
            },

            "product_universality": {
                "definition": "You are measuring 'Product Universality.' This construct evaluates the breadth of the product's target demographic. It measures the continuum between a highly specialized, restrictive product requiring specific hobbies or niche traits, up to a highly accessible, everyday product that appeals broadly to the general mass market.",
                "rubric": "1: Extremely Narrow\n2: Narrow\n3: Somewhat Narrow\n4: Moderate\n5: Somewhat Broad\n6: Broad\n7: Extremely Broad"
            }
        }
    }
}

# New system prompt
diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG,
    suite_description="Prompt suite with continuous anchors (No BARS), including requirements,scale 7, system prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
diagnostic_suite.save(
    suite_folder, filename="continuous_noBARS_wrequirements_scale7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\continuous_noBARS_wrequirements_scale7_SPV2_suite_b30bc4372567.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/continuous_noBARS_wrequirements_scale7_SPV2_suite_b30bc4372567.yml')

- Revised (macro) dimensions, scale 7, with requirements

Here we define three macro dimensions that are based on the 10-dimensional formative variable we settled upon as our 'main' formative specification. However, this would create a strawman out of the informed holistic prompt, which forces the LLM to aggregate the formative dimensions. 

We consider the 'deal quality' as a complex construct. However, this makes it 'too complex' to adequately compare it to a holistic baseline. Therefore, we move up one abstraction layer, and define three macro pillars: asset value, creator cost, communication efficacy. 

In [13]:
EVALUATION_CONFIG_MACRO_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "informed_holistic"],
        "dimensions": {
            "holistic_reward_attractiveness": {
                "definition": "Evaluate the overall attractiveness and subjective desirability of the reward from the perspective of a professional creator. Consider the 'Gestalt' of the offering: does the combination of the brand's reputation, the product's value, and its aesthetic appeal feel like a highly desirable opportunity?",
                "rubric": "1: Extremely Unattractive Reward\n2: Very Unattractive\n3: Unattractive\n4: Neutral/Average Reward\n5: Attractive\n6: Very Attractive\n7: Highly Desirable/Premium Reward"
            },
            "holistic_burden_acceptability": {
                "definition": "Evaluate the subjective acceptability of the demands placed on the creator. Consider the 'Gestalt' of the friction: does the combination of the physical workload and the creative restrictions feel like an unreasonable burden, or a totally acceptable, frictionless ask?",
                "rubric": "1: Completely Unacceptable/Exploitative Burden\n2: Highly Unreasonable\n3: Somewhat Unreasonable\n4: Standard/Acceptable Burden\n5: Light Burden\n6: Very Light/Easy Ask\n7: Completely Frictionless"
            },
            "holistic_pitch_professionalism": {
                "definition": "Evaluate the overall professionalism and subjective 'vibe' of the communication. Consider the 'Gestalt' of the pitch: does the combination of clarity, realism, and enthusiasm make you feel like you are dealing with a highly professional, trustworthy partner?",
                "rubric": "1: Extremely Unprofessional/Sketchy\n2: Very Unprofessional\n3: Unprofessional\n4: Average/Standard Pitch\n5: Professional\n6: Highly Professional\n7: Exceptionally Professional and Trustworthy"
            }
        }
    }
}

# New system prompt
diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_MACRO_FORMATIVE,
    suite_description="Formative Macro NoBARS, revised dimensions, including requirements,scale 7, revision2, system prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
diagnostic_suite.save(
    suite_folder, filename="formative_macro_v2_continuous_noBARS_wrequirements_scale7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\formative_macro_v2_continuous_noBARS_wrequirements_scale7_SPV2_suite_698a82f5d8a3.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/formative_macro_v2_continuous_noBARS_wrequirements_scale7_SPV2_suite_698a82f5d8a3.yml')

- Revised (micro) dimensions, scale 7, with requirements

This is the micro-dimensional prompt. Above we defined three macro categories: asset value, creator cost, and communication efficacy. They are based on the 10 dimensions defined below.

In [ ]:
# "formative_creator_brand_alignment": {
#     "definition": "Evaluate the degree to which the product or service naturally fits an aspirational, high-quality social media creator aesthetic. This measures the organic visual and thematic synergy between the offering and professional creator standards.",
#     "rubric": "1: Extremely Poor Alignment\n2: Poor Alignment\n3: Slightly Poor Alignment\n4: Neutral\n5: Good Alignment\n6: Very Good Alignment\n7: Perfect/Aspirational Alignment"
# },

In [6]:
EVALUATION_CONFIG_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "formative"],
        "dimensions": {
            "formative_brand_prestige": {
                "definition": "Evaluate the established market presence, corporate trust, and overall footprint associated with the brand entity offering the deal.",
                "rubric": "1: Unknown/Suspicious\n2: Very Low\n3: Low\n4: Average/Niche\n5: High\n6: Very High\n7: Top-Tier/Global Leader"
            },
            "formative_perceived_economic_value": {
                "definition": "Evaluate the estimated retail value and premium nature of the physical product, service, or compensation being offered to the creator (the reward).",
                "rubric": "1: Negligible Value\n2: Very Low Value\n3: Low Value\n4: Moderate Value\n5: High Value\n6: Very High Value\n7: Exceptional/Luxury Value"
            },
            "formative_creator_brand_alignment": {
                "definition": "Evaluate the reputational safety and endorsement value of the product or service. This measures the continuum between an offering that would actively dilute or clash with a professional creator's brand equity versus an offering that seamlessly integrates into and actively elevates their social media feed.",
                "rubric": "1: Actively Dilutes Brand Equity (Reputational Risk)\n2: Poor Feed Integration\n3: Slightly Awkward Integration\n4: Neutral/Acceptable Integration\n5: Good Feed Integration\n6: Highly Native and Brand Enhancing\n7: Aspirational and Status Elevating"
            },
            "formative_functional_utility": {
                "definition": "Evaluate the practical, everyday usefulness and utilitarian value of the physical product or service being offered. This measures the continuum between a purely decorative, novelty, or hedonic item versus a highly functional necessity that solves a practical daily problem.",
                "rubric": "1: Purely Novelty/Decorative (Zero Practical Utility)\n2: Very Low Utility\n3: Low Utility\n4: Moderate Utility\n5: High Utility\n6: Very High Utility\n7: Essential/Highly Practical Necessity"
            },
            "formative_product_universality": {
                "definition": "Evaluate the mass-market consumer appeal of the physical product or service on offer. This measures the continuum between an item designed for a highly restricted, specialized target audience versus an everyday product with universal utility across a broad consumer base.",
                "rubric": "1: Extremely Niche/Restrictive\n2: Very Niche\n3: Somewhat Niche\n4: Moderate Appeal\n5: Broad Appeal\n6: Very Broad Appeal\n7: Universal Mass-Market Appeal"
            },
            "formative_workload_volume": {
                "definition": "Evaluate the absolute physical quantity of deliverables and content-creation output required to fulfill the contract. This measures the raw production burden, ranging from a single, low-effort social media post to an exhaustive, multi-asset campaign.",
                "rubric": "1: Near-Zero Workload\n2: Very Light Workload\n3: Light Workload\n4: Moderate Workload\n5: Heavy Workload\n6: Very Heavy Workload\n7: Extreme/Exhaustive Workload"
            },
            "formative_creative_restrictiveness": {
                "definition": "Evaluate the locus of creative control within the collaboration. This measures the continuum between absolute creator autonomy in executing the deliverables versus rigid corporate dictation and structural control.",
                "rubric": "1: Total Creative Freedom\n2: Very High Autonomy\n3: High Autonomy\n4: Moderate Guidelines\n5: Strict Guidelines\n6: Very Strict/Rigid Brief\n7: Total Corporate Dictation"
            },
            "formative_call_to_action_strength": {
                "definition": "Evaluate the relational enthusiasm and cooperative urgency of the pitch. This measures the continuum between a sterile, ambivalent, or purely transactional closing versus a highly motivated, welcoming invitation to collaborate.",
                "rubric": "1: Extremely Ambivalent/Sterile\n2: Very Weak\n3: Weak\n4: Average\n5: Strong\n6: Very Strong\n7: Highly Enthusiastic and Persuasive"
            },
            "formative_rhetorical_realism": {
                "definition": "Evaluate the rhetorical style and tone of the pitch. This measures the continuum between highly inflated, purely positive promotional hyperbole (sales hype, relentless positivity) versus grounded, balanced, and transparent business realism (setting clear expectations, acknowledging practical boundaries or strictness).",
                "rubric": "1: Extreme Promotional Hyperbole (Pure sales hype)\n2: Highly Promotional\n3: Slightly Promotional\n4: Neutral/Standard Pitch\n5: Somewhat Grounded\n6: Highly Grounded and Realistic\n7: Completely Transparent/Factual Realism"
            },
            "formative_proposition_clarity": {
                "definition": "Evaluate the structural clarity and cognitive ease of processing the pitch. This measures the continuum between a disorganized, opaque, or confusing text requiring high cognitive effort to decipher the terms of trade, versus a highly structured, explicit pitch where the required deliverables and offered rewards are immediately obvious.",
                "rubric": "1: Extremely Confusing and Opaque\n2: Very Unclear\n3: Slightly Unclear\n4: Average Clarity\n5: Clear and Structured\n6: Very Clear\n7: Perfectly Explicit and Immediately Obvious"
            },
        }
    }
}

# New system prompt
diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative NoBARS, revised dimensions, including requirements,scale 7, revision2, system prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
diagnostic_suite.save(
    suite_folder, filename="formative_v2_continuous_noBARS_wrequirements_scale7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\formative_v2_continuous_noBARS_wrequirements_scale7_SPV2_suite_f162d59256cc.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/formative_v2_continuous_noBARS_wrequirements_scale7_SPV2_suite_f162d59256cc.yml')

EVEN NEWER VERSION. BEWARE.

In [19]:
EVALUATION_CONFIG_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Here is the Deal Text:\n\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "formative"],
        "dimensions": {
            "brand_prestige": {
                "definition": "Brand Prestige is defined as how well-known and trusted the brand name is in the market. This measures the continuum between a brand that is completely unknown or suspicious, versus a world-famous brand that is a clear global leader in its industry.",
                "rubric": "1: Unknown or Suspicious (No one recognizes this brand name; it feels untrustworthy)\n2: Very Low Recognition\n3: Low Recognition\n4: Average/Niche Recognition (Known only in a very small or specific area)\n5: High Recognition\n6: Very High Recognition\n7: World-Famous/Global Leader (The brand name is recognized by almost everyone)"
            },
            "perceived_economic_value": {
                "definition": "Perceived Economic Value is defined as the estimated retail value and premium nature of the product, service, or compensation offered to the creator. This measures the continuum between a reward of negligible, near-zero financial worth, versus a highly expensive, premium, or luxury offering.",
                "rubric": "1: Negligible Value (A very cheap or nearly worthless offering)\n2: Very Low Value\n3: Low Value\n4: Moderate Value (A standard, average-priced product or service)\n5: High Value\n6: Very High Value\n7: Exceptional/Luxury Value (A highly expensive or premium reward)"
            },
            "creator_brand_alignment": {
                "definition": "Creator Brand Alignment is defined as how naturally the product blends into a professional content creator's social media feed. This measures the continuum between a product that visually or conceptually clashes and looks completely out of place, versus a product that feels like a seamless, natural addition to the creator's curated image.",
                "rubric": "1: Complete Clash (The product looks completely out of place and visually or conceptually jarring)\n2: Poor Fit\n3: Slightly Awkward Fit\n4: Neutral Fit (A standard, acceptable product that neither clashes nor perfectly blends)\n5: Good Fit\n6: High Harmony\n7: Perfect Harmony (A seamless, completely natural addition to a curated image)"
            },
            "functional_utility": {
                "definition": "Functional Utility is defined as the practical, everyday usefulness and utilitarian value of the physical product or service being offered. This measures the continuum between a purely decorative, novelty, or hedonic item versus a highly functional necessity that solves a practical daily problem.",
                "rubric": "1: Purely Novelty/Decorative (Zero practical utility or everyday use)\n2: Very Low Utility\n3: Low Utility\n4: Moderate Utility (A standard item with some practical, but not strictly essential, daily use)\n5: High Utility\n6: Very High Utility\n7: Essential Necessity (A highly practical item that solves a clear, unavoidable daily problem)"
            },
            "product_universality": {
                "definition": "Product Universality is defined as the mass-market consumer appeal of the physical product or service on offer. This measures the continuum between an item designed for a highly restricted, specialized target audience versus an everyday product with universal utility across a broad consumer base.",
                "rubric": "1: Extremely Niche (Highly restricted to a very specialized or rare target audience)\n2: Very Niche\n3: Somewhat Niche\n4: Moderate Appeal (A standard product with average, everyday consumer appeal)\n5: Broad Appeal\n6: Very Broad Appeal\n7: Universal Mass-Market Appeal (An everyday item with near-universal relevance to the general public)"
            },
            "workload_volume": {
                "definition": "Workload Volume is defined as the total time and effort required to execute the collaboration. This measures the continuum between a rapid, low-effort task requiring minimal time and energy, versus a demanding commitment requiring extensive time and high intensity effort.",
                "rubric": "1: Minimal Effort (Negligible time and energy required)\n2: Very Light Effort\n3: Light Effort\n4: Moderate Effort (A standard, manageable investment of time and energy)\n5: Heavy Effort\n6: Very Heavy Effort\n7: Maximum Effort (Extensive time and high intensity energy investment)"
            },
            "creative_restrictiveness": {
                "definition": "Creative Restrictiveness is defined as the level of brand control over the creative process. This measures the continuum between total freedom for the creator to choose their own style and ideas, versus strict rules where the brand decides exactly how the content must look and sound.",
                "rubric": "1: Total Freedom (The creator has full control over the style and ideas)\n2: Very High Freedom\n3: High Freedom\n4: Moderate Rules (A standard balance of freedom and brand guidelines)\n5: Strict Rules\n6: Very Strict Rules\n7: Total Brand Control (The brand decides every detail of the content)"
            },
            "call_to_action_strength": {
                "definition": "Call-to-Action Strength is defined as the level of excitement and warmth in the brand's invitation to collaborate. This measures the continuum between a cold, distant, or purely business-like closing, versus a very welcoming and enthusiastic invitation to work together.",
                "rubric": "1: Extremely Cold and Distant (Purely business-like; no warmth)\n2: Very Weak Excitement\n3: Weak Excitement\n4: Neutral/Average Invitation (A standard, professional closing)\n5: Strong Excitement\n6: Very Strong Excitement\n7: Maximum Excitement and Warmth (A very welcoming and enthusiastic invitation)"
            },
            "rhetorical_objectivity": {
                "definition": "Rhetorical Objectivity is defined as the level of factual realism versus promotional marketing hype in the pitch. This measures the continuum between a text that relies entirely on exaggerated claims, heavy enthusiasm, and sales hype, versus a text that is completely literal, neutral, and strictly focused on business facts.",
                "rubric": "1: Pure Promotional Hyperbole (Entirely driven by sales hype, exaggerated claims, and heavy enthusiasm)\n2: Highly Promotional\n3: Moderately Promotional\n4: Neutral/Standard Pitch (A standard balance of normal marketing warmth and clear factual details)\n5: Grounded and Pragmatic\n6: Highly Objective and Realistic\n7: Purely Factual and Literal (Strictly focused on business facts and constraints with zero promotional hype)"
            },
            "proposition_clarity": {
                "definition": "Proposition Clarity is defined as the structural clarity and cognitive ease of processing the pitch. This measures the continuum between a disorganized, vague, or confusing text requiring high cognitive effort to decipher the terms of trade, versus a highly structured, explicit pitch where the required deliverables and offered rewards are immediately obvious.",
                "rubric": "1: Extremely Confusing and Opaque (Highly disorganized; the required deliverables and rewards are very difficult to decipher)\n2: Very Unclear\n3: Slightly Unclear\n4: Average Clarity (A standard, readable pitch where the main terms are understandable with normal cognitive effort)\n5: Clear and Structured\n6: Very Clear\n7: Perfectly Explicit and Immediately Obvious (Highly structured; the exact deliverables and rewards are instantly clear with zero cognitive effort)"
            }
        }
    }
}

# New system prompt
diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative NoBARS, revised dimensions, including requirements,scale 7, revision2, system prompt V3, user prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW2)
diagnostic_suite.save(
    suite_folder, filename="v2_continuous_noBARS_wrequirements_scale7_SPV3_UPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\v2_continuous_noBARS_wrequirements_scale7_SPV3_UPV2_suite_1e1ff82a28e2.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/v2_continuous_noBARS_wrequirements_scale7_SPV3_UPV2_suite_1e1ff82a28e2.yml')

### BARS Formative

In [60]:
EVALUATION_CONFIG = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "aspirational_framing": {
                "definition": """how well the text elevates a mundane or standard product into a highly desirable, aesthetic lifestyle experience that a creator would want to associate with their personal brand.""",
                "rubric": """1: Dry & Functional (Describes the product strictly by its physical traits or utility, reading like a catalog entry. E.g., 'We sell scented cleaning products').
2: Mildly Appealing (Hints at a positive outcome or feeling, but lacks immersive storytelling or strong lifestyle cues).
3: Lifestyle-Oriented (Actively frames the product within a relatable, aesthetically pleasing scenario, e.g., 'Perfect for your cozy autumn evenings').
4: Highly Aspirational (Masterfully sells a vibe, feeling, or elevated status. It makes the product feel like an essential part of an idealized, 'high-end' or romanticized lifestyle, making the creator want to live that narrative)."""
            },
            "creator_validation": {
                "definition": """whether the brand treats the creator as a respected creative partner versus an interchangeable advertising metric.""",
                "rubric": """1: Transactional / Cattle Call (Addresses a generic audience, reads like a standard consumer ad, or demands labor without acknowledging the creator's skill or niche).
2: Standard Brief (Polite but clinical. Lacks personal warmth or recognition of the creator's unique value).
3: Collaborative & Welcoming (Uses inclusive, relational language. Acknowledges the creator's specific niche, e.g., 'Perfect for creators who share beauty routines').
4: Highly Validating & Exclusive (Strokes the creator's ego. Frames the deal as a selective partnership, validating their expertise, aesthetic eye, or community influence. Makes them feel hand-picked and highly valued)."""
            },
            "creative_autonomy": {
                "definition": """how much psychological ownership the creator feels they will have over the content, versus feeling restricted by a rigid corporate brief.""",
                "rubric": """1: Highly Prescriptive / Restrictive (Demands strict adherence to specific talking points, scripts, or leaves no room for personal interpretation).
2: Ambiguous Autonomy (Mentions deliverables but is unclear about how much creative freedom the creator actually has).
3: Encourages Authenticity (Explicitly asks the creator to share their genuine experience, personal routine, or honest review).
4: Full Creative Ownership (Actively champions the creator's unique style. Uses phrases like 'in your own unique way,' 'your authentic voice,' or encourages them to build a personal story around the product)."""
            },
            "cognitive_friction": {
                "definition": """the psychological weight of parsing the deal. High friction means the creator has to hunt for deliverables, value, or location requirements, causing decision fatigue. LOW friction means the text is highly structured for a rapid 'give and get' assessment.""",
                "rubric": """1: High Friction (Dense blocks of text, buried requirements, vague compensation, or missing crucial logistics like location requirements).
2: Moderate Friction (Information is present but requires reading through standard paragraphs to find the exact deliverables and rewards).
3: Low Friction (Uses clear spacing or lists to separate the product description from the campaign requirements).
4: Zero Friction / Instantly Scannable (Masterful use of formatting—bullet points, bold text, emojis—to instantly communicate exactly what the creator must do and exactly what they receive in return. Decision can be made in 3 seconds)."""
            }
        }
    }
}

In [ ]:
# # Build and save the suite
# diagnostic_suite = build_formative_suite(
#     EVALUATION_CONFIG, "Diagnostic Gatekeeper suite for Negative Binomial regression")
# diagnostic_suite.save(suite_folder, filename="diagnostic_gatekeeper_easter")

TypeError: build_formative_suite() missing 2 required positional arguments: 'scale_size' and 'system_prompt_template'

## Holistic

### Naive baseline

- Scale 7, subjective phrasing (a posteriori trait)

In [ ]:
EVALUATION_CONFIG_NAIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "overall_deal_attractiveness": {
                "definition": "This evaluates the overall appeal of the barter opportunity from the perspective of a typical content creator. It assesses the total subjective value of the offer, ranging from a fundamentally undesirable deal that offers no compelling reward (Extremely Weak), up to a highly coveted, premium opportunity that is instantly desirable (Excellent).",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Slightly Weak\n4: Fair / Average\n5: Slightly Strong\n6: Strong\n7: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Baseline, scale 7, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(suite_folder, filename="holistic_naive_S7_SPV2")

- Scale 7, objective phrasing (a priori trait)

In [71]:
EVALUATION_CONFIG_NAIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "overall_deal_quality": {
                "definition": "This evaluates the overall economic equity and holistic quality of the barter exchange. It assesses the total balance between the required deliverables and the offered product—ranging from a fundamentally imbalanced, low-quality offer (Extremely Weak), up to an exceptionally balanced, high-quality, and premium opportunity (Excellent).",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Slightly Weak\n4: Fair / Average\n5: Slightly Strong\n6: Strong\n7: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Quality, scale 7, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(suite_folder, filename="holistic_naive_quality_S7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_naive_quality_S7_SPV2_suite_722bbd35b498.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_naive_quality_S7_SPV2_suite_722bbd35b498.yml')

- scale 7, includes requirements

In [ ]:
EVALUATION_CONFIG_NAIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "overall_deal_quality": {
                "definition": "This evaluates the overall economic equity and holistic quality of the barter exchange. It assesses the total balance between the required deliverables and the offered product—ranging from a fundamentally imbalanced, low-quality offer (Extremely Weak), up to an exceptionally balanced, high-quality, and premium opportunity (Excellent).",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Slightly Weak\n4: Fair / Average\n5: Slightly Strong\n6: Strong\n7: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Quality, includes requirements, scale 7, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(
    suite_folder, filename="holistic_naive_quality_wrequirements_S7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_naive_quality_wrequirements_S7_SPV2_suite_4734896e1e08.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_naive_quality_wrequirements_S7_SPV2_suite_4734896e1e08.yml')

### Informed baseline

- scale 7, subjective phrasing


In [67]:
EVALUATION_CONFIG_INFORMED = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "informed_deal_attractiveness": {
                "definition": "This evaluates the overall appeal of the barter opportunity from the perspective of a typical content creator. To determine this overall attractiveness, you must simultaneously consider and synthesize six underlying factors into a single judgment:\n1. Brand Recognition: How established the company is.\n2. Creator Brand Alignment: How well the product fits an aspirational creator aesthetic.\n3. Product Universality: The broad mass-market appeal of the actual product.\n4. Workload Volume: The amount of effort and content required from the creator.\n5. Call-to-Action Strength: The clarity and persuasive power of the next steps.\n6. Value Proposition Clarity: How well the exchange is articulated.\n\nAssess the total subjective value of the offer based on these aggregated factors—ranging from a fundamentally undesirable deal (Extremely Weak), up to a highly coveted, premium opportunity (Excellent).",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Slightly Weak\n4: Fair / Average\n5: Slightly Strong\n6: Strong\n7: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_INFORMED,
    suite_description="Holistic Informed Baseline, scale 7, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(suite_folder, filename="holistic_informed_S7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_informed_S7_SPV2_suite_e403c8a79195.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_informed_S7_SPV2_suite_e403c8a79195.yml')

- scale 7, objective phrasing

In [72]:
EVALUATION_CONFIG_INFORMED = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "informed_deal_quality": {
                "definition": "This evaluates the overall economic equity and holistic quality of the barter exchange. To determine this overall quality, you must simultaneously consider and synthesize six underlying factors into a single judgment:\n1. Brand Recognition: How established the company is.\n2. Creator Brand Alignment: How well the product fits an aspirational creator aesthetic.\n3. Product Universality: The broad mass-market appeal of the actual product.\n4. Workload Volume: The amount of effort and content required from the creator.\n5. Call-to-Action Strength: The clarity and persuasive power of the next steps.\n6. Value Proposition Clarity: How well the exchange is articulated.\n\nAssess the total balance and structural quality of the offer based on these aggregated factors—ranging from a fundamentally imbalanced, low-quality offer (Extremely Weak), up to an exceptionally balanced, high-quality, and premium opportunity (Excellent).",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Slightly Weak\n4: Fair / Average\n5: Slightly Strong\n6: Strong\n7: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_INFORMED,
    suite_description="Holistic Informed Quality, scale 7, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(suite_folder, filename="holistic_informed_quality_S7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_informed_quality_S7_SPV2_suite_1dd96c436d00.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_informed_quality_S7_SPV2_suite_1dd96c436d00.yml')

- scale 7, objective phrasing, includes requirements

In [ ]:
EVALUATION_CONFIG_INFORMED = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "informed_deal_quality": {
                "definition": "This evaluates the overall economic equity and holistic quality of the barter exchange. To determine this overall quality, you must simultaneously consider and synthesize six underlying factors into a single judgment:\n1. Brand Recognition: How established the company is.\n2. Creator Brand Alignment: How well the product fits an aspirational creator aesthetic.\n3. Product Universality: The broad mass-market appeal of the actual product.\n4. Workload Volume: The amount of effort and content required from the creator.\n5. Call-to-Action Strength: The clarity and persuasive power of the next steps.\n6. Value Proposition Clarity: How well the exchange is articulated.\n\nAssess the total balance and structural quality of the offer based on these aggregated factors—ranging from a fundamentally imbalanced, low-quality offer (Extremely Weak), up to an exceptionally balanced, high-quality, and premium opportunity (Excellent).",
                "rubric": "1: Extremely Weak\n2: Weak\n3: Slightly Weak\n4: Fair / Average\n5: Slightly Strong\n6: Strong\n7: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_INFORMED,
    suite_description="Holistic Informed Quality, scale 7, includes requirements, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(
    suite_folder, filename="holistic_informed_quality_wrequirements_S7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_informed_quality_wrequirements_S7_SPV2_suite_a1170ceef97e.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_informed_quality_wrequirements_S7_SPV2_suite_a1170ceef97e.yml')

- scale 7, objective phrasing, includes requirements, adapted to the new 10-dimensional formative prompt

In [12]:
EVALUATION_CONFIG_INFORMED = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "informed_holistic"],
        "dimensions": {
            "holistic_deal_attractiveness": {
                "definition": "Evaluate the overall attractiveness, fairness, and quality of this barter deal from the perspective of a professional content creator. Consider the 'Gestalt' or total balance of the offer: is the perceived reward (the product and brand) worth the required cost (the workload and rules)?",
                "rubric": "1: Extremely Unattractive/Exploitative\n2: Very Unattractive\n3: Unattractive\n4: Neutral/Average Deal\n5: Attractive\n6: Very Attractive\n7: Extremely Attractive/Premium Deal"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_INFORMED,
    suite_description="Holistic Informed Quality V2, scale 7, includes requirements, SPV2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_NEW)
holistic_suite.save(
    suite_folder, filename="holistic_informed_quality_V2_wrequirements_S7_SPV2")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_informed_quality_V2_wrequirements_S7_SPV2_suite_6faa8a8b031c.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_informed_quality_V2_wrequirements_S7_SPV2_suite_6faa8a8b031c.yml')

- scale 4, old system prompt

In [ ]:
EVALUATION_CONFIG = {
    "deal_pitch": {
        "human_name": "Holistic Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "holistic_deal_attractiveness": {
                "definition": "You are measuring 'Holistic Deal Attractiveness.' This construct evaluates the overall, generalized appeal and economic viability of the barter opportunity from the perspective of a typical content creator. It measures the continuum between a fundamentally unappealing offer with a deeply negative subjective value exchange that strongly discourages participation, up to a highly coveted, premium opportunity with an overwhelmingly positive value exchange.",
                "rubric": "1: Weak\n2: Fair\n3: Good\n4: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG,
    suite_description="Holistic Baseline Test",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_OLD)
holistic_suite.save(suite_folder, filename="holistic_baseline_continuum")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\holistic_baseline_continuum_suite_0eaeb6c665d8.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/holistic_baseline_continuum_suite_0eaeb6c665d8.yml')

## Brand recognition holistic

In [ ]:
EVALUATION_CONFIG = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Deal Pitch:\nTitle: {deal_title}\n\nBody:\n{deal_text}",
        "tags": ["deal_pitch"],
        "dimensions": {
            "brand_recognition": {
                "definition": "how widely recognized and established the specific brand or company is to the general public, strictly independent of how familiar or common the actual product is (e.g., a generic smartphone vs. an Apple iPhone).",
                "rubric": """1: Not Known / Anonymous (The brand name is entirely missing, irrelevant, or appears to be a generic white-label/dropshipping operation. Zero inherent brand equity).
2: Somewhat Unknown / Local (A distinct, real business—such as a local restaurant, independent webshop, or new startup—but it lacks widespread mainstream recognition. The average consumer has likely never heard of it).
3: Somewhat Known / Established (A solid regional or national brand. Consumers might recognize the name from high-street presence, targeted ads, or it is a highly established player within a specific consumer niche).
4: Well-Known / Household Name (A ubiquitous, top-tier national or global brand. Instant, universal recognition by the general public, bringing inherent social proof to the creator)."""
            }
        }
    }
}

In [ ]:
# Build and save the suite
diagnostic_suite = build_formative_suite(
    EVALUATION_CONFIG, "Brand Recognition, single dimension test")
diagnostic_suite.save(suite_folder, filename="brand_recognition_first_test")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\BARTER_DEALS\brand_recognition_first_test_suite_26fac938d6ea.yml


WindowsPath('../prompts/PromptSuites/sandbox/BARTER_DEALS/brand_recognition_first_test_suite_26fac938d6ea.yml')

## GARBAGE BIN

- I generated data with these prompts and analysed it, but it didnt really do much compared to the previous one, so it can be IGNORED

In [ ]:
SYSTEM_PROMPT_BARTER_NEW = """You are an expert evaluator of influencer marketing deals ("Deal Texts") for Barter, an app-based creator marketplace. 
In this ecosystem, creators scroll a feed of deals and apply to collaborate. If accepted, they produce content (e.g., UGC, TikToks, Instagram Reels) primarily in exchange for the brand's offering, with occasional cash supplements. Your evaluation must be grounded in this creator ecosystem.

You will be provided with a Deal Text consisting of a Title, Body, and Requirements. Please analyze this text, focusing strictly on the following dimension: **{dim_key_clean}**.

Dimension Definition:
{definition}

Scale Definition:
{rubric}
"""

#### Formative

In [ ]:


EVALUATION_CONFIG_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Here is the Deal Text:\n\nTitle: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "formative"],
        "dimensions": {
            "brand_prestige": {
                "definition": "Brand Prestige is defined as how well-known and trusted the brand name is in the market. This measures the continuum between a brand that is completely unknown or suspicious, versus a world-famous brand that is a clear global leader in its industry.",
                "rubric": "1: Unknown or Suspicious (No one recognizes this brand name; it feels untrustworthy)\n2: Very Low Recognition\n3: Low Recognition\n4: Average/Niche Recognition (Known only in a very small or specific area)\n5: High Recognition\n6: Very High Recognition\n7: World-Famous/Global Leader (The brand name is recognized by almost everyone)"
            },
            "perceived_economic_value": {
                "definition": "Perceived Economic Value is defined as the estimated retail value and premium nature of the product, service, or compensation offered to the creator. This measures the continuum between a reward of negligible, near-zero financial worth, versus a highly expensive, premium, or luxury offering.",
                "rubric": "1: Negligible Value (A very cheap or nearly worthless offering)\n2: Very Low Value\n3: Low Value\n4: Moderate Value (A standard, average-priced product or service)\n5: High Value\n6: Very High Value\n7: Exceptional/Luxury Value (A highly expensive or premium reward)"
            },
            "creator_brand_alignment": {
                "definition": "Creator Brand Alignment is defined as how naturally the product blends into a professional content creator's social media feed. This measures the continuum between a product that visually or conceptually clashes and looks completely out of place, versus a product that feels like a seamless, natural addition to the creator's curated image.",
                "rubric": "1: Complete Clash (The product looks completely out of place and visually or conceptually jarring)\n2: Poor Fit\n3: Slightly Awkward Fit\n4: Neutral Fit (A standard, acceptable product that neither clashes nor perfectly blends)\n5: Good Fit\n6: High Harmony\n7: Perfect Harmony (A seamless, completely natural addition to a curated image)"
            },
            "functional_utility": {
                "definition": "Functional Utility is defined as the practical, everyday usefulness and utilitarian value of the physical product or service being offered. This measures the continuum between a purely decorative, novelty, or hedonic item versus a highly functional necessity that solves a practical daily problem.",
                "rubric": "1: Purely Novelty/Decorative (Zero practical utility or everyday use)\n2: Very Low Utility\n3: Low Utility\n4: Moderate Utility (A standard item with some practical, but not strictly essential, daily use)\n5: High Utility\n6: Very High Utility\n7: Essential Necessity (A highly practical item that solves a clear, unavoidable daily problem)"
            },
            "product_universality": {
                "definition": "Product Universality is defined as the mass-market consumer appeal of the physical product or service on offer. This measures the continuum between an item designed for a highly restricted, specialized target audience versus an everyday product with universal utility across a broad consumer base.",
                "rubric": "1: Extremely Niche (Highly restricted to a very specialized or rare target audience)\n2: Very Niche\n3: Somewhat Niche\n4: Moderate Appeal (A standard product with average, everyday consumer appeal)\n5: Broad Appeal\n6: Very Broad Appeal\n7: Universal Mass-Market Appeal (An everyday item with near-universal relevance to the general public)"
            },
            "workload_volume": {
                "definition": "Workload Volume is defined as the total time and effort required to execute the collaboration. This measures the continuum between a rapid, low-effort task requiring minimal time and energy, versus a demanding commitment requiring extensive time and high intensity effort.",
                "rubric": "1: Minimal Effort (Negligible time and energy required)\n2: Very Light Effort\n3: Light Effort\n4: Moderate Effort (A standard, manageable investment of time and energy)\n5: Heavy Effort\n6: Very Heavy Effort\n7: Maximum Effort (Extensive time and high intensity energy investment)"
            },
            "creative_restrictiveness": {
                "definition": "Creative Restrictiveness is defined as the level of brand control over the creative process. This measures the continuum between total freedom for the creator to choose their own style and ideas, versus strict rules where the brand decides exactly how the content must look and sound.",
                "rubric": "1: Total Freedom (The creator has full control over the style and ideas)\n2: Very High Freedom\n3: High Freedom\n4: Moderate Rules (A standard balance of freedom and brand guidelines)\n5: Strict Rules\n6: Very Strict Rules\n7: Total Brand Control (The brand decides every detail of the content)"
            },
            "call_to_action_strength": {
                "definition": "Call-to-Action Strength is defined as the level of excitement and warmth in the brand's invitation to collaborate. This measures the continuum between a cold, distant, or purely business-like closing, versus a very welcoming and enthusiastic invitation to work together.",
                "rubric": "1: Extremely Cold and Distant (Purely business-like; no warmth)\n2: Very Weak Excitement\n3: Weak Excitement\n4: Neutral/Average Invitation (A standard, professional closing)\n5: Strong Excitement\n6: Very Strong Excitement\n7: Maximum Excitement and Warmth (A very welcoming and enthusiastic invitation)"
            },
            "rhetorical_objectivity": {
                "definition": "Rhetorical Objectivity is defined as the level of factual realism versus promotional marketing hype in the pitch. This measures the continuum between a text that relies entirely on exaggerated claims, heavy enthusiasm, and sales hype, versus a text that is completely literal, neutral, and strictly focused on business facts.",
                "rubric": "1: Pure Promotional Hyperbole (Entirely driven by sales hype, exaggerated claims, and heavy enthusiasm)\n2: Highly Promotional\n3: Moderately Promotional\n4: Neutral/Standard Pitch (A standard balance of normal marketing warmth and clear factual details)\n5: Grounded and Pragmatic\n6: Highly Objective and Realistic\n7: Purely Factual and Literal (Strictly focused on business facts and constraints with zero promotional hype)"
            },
            "proposition_clarity": {
                "definition": "Proposition Clarity is defined as the structural clarity and cognitive ease of processing the pitch. This measures the continuum between a disorganized, vague, or confusing text requiring high cognitive effort to decipher the terms of trade, versus a highly structured, explicit pitch where the required deliverables and offered rewards are immediately obvious.",
                "rubric": "1: Extremely Confusing and Opaque (Highly disorganized; the required deliverables and rewards are very difficult to decipher)\n2: Very Unclear\n3: Slightly Unclear\n4: Average Clarity (A standard, readable pitch where the main terms are understandable with normal cognitive effort)\n5: Clear and Structured\n6: Very Clear\n7: Perfectly Explicit and Immediately Obvious (Highly structured; the exact deliverables and rewards are instantly clear with zero cognitive effort)"
            }
        }
    }
}

# New system prompt
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative NoBARS, revised dimensions, including requirements,scale 7, revision2, system prompt V4, user prompt V2",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER_NEW)
formative_suite.save(
    suite_folder, filename="formative_v2_continuous_noBARS_wrequirements_scale7_SPV4_UPV2")

#### Holistic informed

In [ ]:
EVALUATION_CONFIG_MACRO_FORMATIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "informed_holistic"],
        "dimensions": {
            "overall_reward_value": {
                "definition": "Overall Reward Value is defined as the total desirability and worth of the compensation package being offered to the creator. This measures the continuum between a highly unappealing reward with little to no worth, versus an exceptionally premium, highly coveted offering.",
                "rubric": "1: Highly Unappealing (A very poor offering with negligible worth or desirability)\n2: Very Low Appeal\n3: Low Appeal\n4: Standard Market Value (An average, typical compensation package)\n5: High Appeal\n6: Very High Appeal\n7: Highly Premium Offering (An exceptionally valuable and highly coveted reward)"
            },
            "overall_task_demand": {
                "definition": "Overall Task Demand is defined as the total amount of effort, time, and rigid compliance required to complete the campaign. This measures the continuum between a very fast, easy, and flexible request, versus a highly demanding, time-consuming project with strict rules.",
                "rubric": "1: Minimal Demand (An extremely fast, easy task with total flexibility)\n2: Very Low Demand\n3: Low Demand\n4: Moderate Demand (A standard campaign requiring an average amount of time and effort)\n5: High Demand\n6: Very High Demand\n7: Extreme Demand (A highly time-consuming project with intensive requirements and strict rules)"
            },
            "overall_pitch_quality": {
                "definition": "Overall Pitch Quality is defined as the clarity, structure, and professional tone of the brand's written communication. This measures the continuum between a highly confusing, poorly written, or unprofessional text, versus a flawlessly structured, perfectly clear, and highly professional proposal.",
                "rubric": "1: Extremely Poor Quality (The text is highly confusing, unstructured, or deeply unprofessional)\n2: Very Low Quality\n3: Low Quality\n4: Standard Quality (A moderately clear, average business communication)\n5: High Quality\n6: Very High Quality\n7: Exceptional Quality (A perfectly clear, flawlessly structured, and highly professional proposal)"
            }
        }
    }
}


diagnostic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_MACRO_FORMATIVE,
    suite_description="Informed holistic NoBARS, revised dimensions, including requirements,scale 7, revision3, system prompt V4",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER_NEW)
diagnostic_suite.save(
    suite_folder, filename="holistic_informed_v2_continuous_noBARS_wrequirements_scale7_SPV4")

#### Holistic naive

In [ ]:
EVALUATION_CONFIG_NAIVE = {
    "deal_pitch": {
        "human_name": "Deal Pitch",
        "user_template": "Title: {deal_title}\n\nBody:\n{deal_text}\n\nRequirements:\n{creators_requirement}",
        "tags": ["deal_pitch", "naive_holistic"],
        "dimensions": {
            "overall_deal_attractiveness": {
                "definition": "This evaluates the overall appeal of the barter opportunity from the perspective of a typical content creator. It assesses the total subjective value of the offer, ranging from a fundamentally undesirable deal that offers no compelling reward, up to a highly coveted, premium opportunity that is instantly desirable.",
                "rubric": "1: Extremely Weak (A fundamentally undesirable deal offering no compelling value or appeal)\n2: Weak\n3: Slightly Weak\n4: Fair / Average (A standard, typical deal with baseline market appeal)\n5: Slightly Strong\n6: Strong\n7: Excellent (A highly coveted, premium opportunity that is instantly desirable)"
            }
        }
    }
}
# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Quality, scale 7, SPV4",
    scale_size=7,
    system_prompt_template=SYSTEM_PROMPT_BARTER_NEW)
holistic_suite.save(suite_folder, filename="holistic_naive_quality_S7_SPV4")

# FeedbackQA

In [6]:
suite_folder = Path("../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK")
pm = PromptManager(suite_folder)

PromptManager initialized with folder: ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK


In [4]:
SYSTEM_PROMPT_QA = """You are an expert evaluator of Question and Answer (Q&A) pairs for a public health information platform. 
In this ecosystem, individuals ask questions regarding the COVID-19 pandemic, and an informational text passage is provided as the answer. Your evaluation must be grounded in this specific context of health-related information exchange.

Your task is to evaluate the provided {human_name} conditional strictly on its {dim_key_clean}.
{definition}

Scale definition:
{rubric}

Output only the integer score."""

## Holistic

### Naive baseline

- Scale 4, subjective phrasing (a posteriori trait)

In [ ]:
EVALUATION_CONFIG_NAIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa"],
        "dimensions": {
            "overall_answer_quality": {

                "definition": "This evaluates the overall quality and helpfulness of the answer provided in response to the user's question. It assesses the total subjective value of the response, ranging from a fundamentally unhelpful, irrelevant, or incorrect answer (Bad), up to a highly effective, accurate, and perfect response (Excellent).",

                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"

            }
        }
    }
}


# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Baseline FeedbackQA, scale 4, SPV1, subjective",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA
)
holistic_suite.save(suite_folder, filename="holistic_naive_S4_SPV1_subjective")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK\holistic_naive_S4_SPV1_subjective_suite_770268391b2b.yml


WindowsPath('../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK/holistic_naive_S4_SPV1_subjective_suite_770268391b2b.yml')

- Scale 4, objective phrasing (a priori trait)

In [10]:
EVALUATION_CONFIG_NAIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa"],
        "dimensions": {
            "overall_answer_quality": {
                "definition": "This evaluates the overall objective informational quality and structural integrity of the answer provided in response to the user's question. It assesses the total informational value and composition of the text—ranging from a fundamentally deficient, low-quality answer with severely degraded informational and structural value (Bad), up to an exceptionally sound, high-quality answer with perfect informational and structural integrity (Excellent).",
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            }
        }
    }
}

# Build and save the suite
holistic_suite = build_formative_suite(
    config=EVALUATION_CONFIG_NAIVE,
    suite_description="Holistic Naive Baseline FeedbackQA, scale 4, SPV1, objective",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA
)
holistic_suite.save(suite_folder, filename="holistic_naive_S4_SPV1_objective")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK\holistic_naive_S4_SPV1_objective_suite_946e242b933f.yml


WindowsPath('../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK/holistic_naive_S4_SPV1_objective_suite_946e242b933f.yml')

### Informed baseline

In [11]:
EVALUATION_CONFIG_INFORMED = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "informed_baseline"],
        "dimensions": {
            "informed_answer_quality": {
                "definition": "This evaluates the overall objective informational quality and structural integrity of the answer provided in response to the user's question. To determine this overall quality, you must simultaneously consider and synthesize four underlying factors into a single judgment:\n1. Topical Relevance: The degree to which the answer directly addresses the core entity and obeys contextual constraints.\n2. Information Completeness: The depth and comprehensiveness of the answer.\n3. Structural Clarity: The compositional logic and syntactic organization of the text.\n4. Actionability & Utility: The functional and practical directives contained within the response.\n\nAssess the total informational value and composition of the text based on these aggregated factors—ranging from a fundamentally deficient, low-quality answer with severely degraded informational and structural value (Bad), up to an exceptionally sound, high-quality answer with perfect informational and structural integrity (Excellent).",
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            }
        }
    }
}

# Build and save the formative suite
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_INFORMED,
    suite_description="Holistic Informed Baseline FeedbackQA, Scale 4, SPV1",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA
)
formative_suite.save(
    suite_folder, filename="holistic_informed_S4_SPV1_objective")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK\holistic_informed_S4_SPV1_objective_suite_be0f22b3913e.yml


WindowsPath('../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK/holistic_informed_S4_SPV1_objective_suite_be0f22b3913e.yml')

## Formative

In [12]:
EVALUATION_CONFIG_FORMATIVE = {
    "qa_pair": {
        "human_name": "Q&A Pair",
        "user_template": "Question:\n{question}\n\nAnswer:\n{answer}",
        "tags": ["qa_pair", "feedback_qa", "formative"],
        "dimensions": {
            "topical_relevance": {
                "definition": "Evaluates the degree to which the answer directly addresses the core entity of the prompt while strictly obeying any established contextual constraints. It measures the continuum between providing completely irrelevant information or ignoring explicit constraints (Bad), up to a precise, perfectly aligned response that directly targets the specific context (Excellent).",
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            },
            "information_completeness": {
                "definition": "Evaluates the depth and comprehensiveness of the answer. It measures the continuum between an overly brief, generalized response missing critical parameters or caveats (Bad), up to a highly comprehensive, multi-faceted answer that fully satisfies the informational requirements of the question (Excellent).",
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            },
            "structural_clarity": {
                "definition": "Evaluates the compositional logic and syntactic organization of the answer. It measures the continuum between dense, poorly formatted blocks of text with chaotic syntax (Bad), up to highly intuitive, perfectly organized information that utilizes clear formatting (such as bullet points) to separate complex ideas (Excellent).",
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            },
            "actionability_and_utility": {
                "definition": "Evaluates the functional and practical directives contained within the response. It measures the continuum between purely theoretical, passive answers that provide no distinct next steps or resources (Bad), up to highly actionable responses that provide explicit resources, functional directives, and clear guidance (Excellent).",
                "rubric": "1: Bad\n2: Could be Improved\n3: Acceptable\n4: Excellent"
            }
        }
    }
}

# Build and save the formative suite
formative_suite = build_formative_suite(
    config=EVALUATION_CONFIG_FORMATIVE,
    suite_description="Formative Decomposition FeedbackQA, Scale 4, SPV1",
    scale_size=4,
    system_prompt_template=SYSTEM_PROMPT_QA
)
formative_suite.save(suite_folder, filename="formative_decomposition_S4_SPV1")

✅ Saved suite to ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK\formative_decomposition_S4_SPV1_suite_1dfe8cb6857a.yml


WindowsPath('../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK/formative_decomposition_S4_SPV1_suite_1dfe8cb6857a.yml')